In [1]:
import pandas as pd
import numpy as np
import pickle
from lightgbm import LGBMRegressor
MODELS_CACHE = {}

In [2]:
df = pd.read_csv("full_data.csv")

In [3]:
pitchers = list(df["pitcherId"].unique())
pitches = list(df["pitchType"].unique())
pitches.remove("KN")
pitches.remove("ST")
lr = ["L", "R"]

features = ['inducedVertBreak', 'horzBreak', 'extension', 'relX', 'relZ', 'releaseVelocity', 'spinRate']
targets = ["rv"]

test = df[df["year"] == 2026]

In [4]:
def pred_rv(test, features, target, pitch, side):
    if (pitch, side) not in MODELS_CACHE:
        with open(f'models/26_{pitch}_{side}.pkl', 'rb') as f:
            MODELS_CACHE[(pitch, side)] = pickle.load(f)

    model = MODELS_CACHE[(pitch, side)]
    X_test = test[features]
    predictions = model.predict(X_test) 
    
    return pd.Series(predictions, index=test.index)

In [5]:
dfs = []

for pitcher in pitchers: 
    for pitch in pitches: 
        for side in lr: 
            te = test[(test["pitcherId"] == pitcher) & (test["pitchType"] == pitch) & (test["pitcherHand"] == side)] 
            if len(te) >= 25: 
                dfx = {} 
                dfx["Pitcher"] = te["Pitcher"].iloc[0]
                dfx["pitcherId"] = pitcher
                dfx["Team"] = te["pitchingTeam"].iloc[0]
                dfx["PitchType"] = pitch 
                dfx["PitcherThrows"] = side 
                dfx["Count"] = len(te) 
                for feature in features: 
                    dfx[feature] = te[feature].mean() 
                for target in targets: 
                    dfz = pd.DataFrame([dfx], columns=features) 
                    dfx["xrv"] = float(pred_rv(dfz, features, target, pitch, side).iloc[0]) 
                dfs.append(dfx)

In [6]:
stuff = pd.DataFrame(dfs).reset_index(drop=True)

In [7]:
stuff[stuff["Team"] == "URI"].sort_values("xrv")

,Pitcher,pitcherId,Team,PitchType,PitcherThrows,Count,inducedVertBreak,horzBreak,extension,relX,relZ,releaseVelocity,spinRate,xrv
2753,E. Maloney,1085415168,URI,CH,R,87,9.350467,9.442642,4.927112,-0.090336,6.935631,75.586409,1500.601278,-0.034565
2754,E. Maloney,1085415168,URI,CU,R,60,-6.452983,-1.332604,4.757064,-0.271923,6.784984,73.773580,1926.535019,0.038602
3211,J. Sabbath,1110853376,URI,SL,R,206,5.095956,-11.401671,5.865138,-1.798176,5.668853,78.623431,2351.838092,0.052849
3199,J. Cullen,1358089441,URI,CU,R,54,-12.875800,-14.190159,5.251227,-1.184985,6.212153,76.218862,2742.228921,0.061995
3198,J. Cullen,1358089441,URI,SL,R,88,2.784822,-4.918909,5.581707,-1.453798,5.853970,83.277734,2592.593725,0.064663
9036,L. Lavigueur,1143021056,URI,SL,R,122,-4.844720,-19.924092,5.285987,-1.489170,5.899264,78.553081,2548.945651,0.067298
3196,J. Cullen,1358089441,URI,FA,R,280,21.846538,10.306399,6.014419,-1.058248,6.122621,91.539084,2256.348520,0.067374
2763,C. Johnston,1241946624,URI,SL,R,71,4.239643,-8.041021,6.044654,-0.780000,6.087559,78.980572,2379.899720,0.067794
2756,C. Maloney,1336247552,URI,SL,R,59,-2.191296,-6.506970,5.328861,-2.294733,5.774931,82.491403,2544.400868,0.068900
3195,D. Asencio,1241946368,URI,SL,R,25,4.573965,-11.880708,5.570063,-1.829854,5.830872,75.195108,2338.561985,0.069998


In [8]:
from scipy import stats

#df['zscore'] = stats.zscore(df['column'])
stuff["Stuff+"] = 100 + (-10 * stuff.groupby("PitchType")["xrv"].transform(stats.zscore))
stuff["Stuff+"] = stuff["Stuff+"].round().astype(int)

In [9]:
stuff.sort_values("Stuff+", ascending=False)

,Pitcher,pitcherId,Team,PitchType,PitcherThrows,Count,inducedVertBreak,horzBreak,extension,relX,relZ,releaseVelocity,spinRate,xrv,Stuff+
2753,E. Maloney,1085415168,URI,CH,R,87,9.350467,9.442642,4.927112,-0.090336,6.935631,75.586409,1500.601278,-0.034565,162
2625,S. Shenosky,1689800810,PSU,CH,R,28,8.252221,16.029555,7.166033,-1.228577,6.089665,75.110317,2057.848403,-0.002247,146
10285,B. Bak,1123967488,UIC,FC,L,33,3.833749,-1.239232,4.741356,0.977097,6.242619,79.076230,2258.488391,-0.090345,146
4768,J. Vanesko,1210138624,BRY,FC,L,41,9.463535,-1.680251,6.344392,1.777137,6.043031,81.595356,2183.780640,-0.082906,144
3016,D. Jackson,1205651242,UCSB,CH,L,138,3.330003,-20.161013,5.784155,1.589700,5.824918,76.643101,2555.609407,0.004463,143
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8103,B. Lanman,1087364096,AC,SL,L,42,5.796549,2.342767,6.954041,1.374516,5.653946,80.019513,1866.446180,0.171337,45
10273,J. Lessman,1334299648,UIC,CU,L,169,-10.793909,8.968286,5.668975,1.968539,5.044650,75.522662,2163.594687,0.195953,45
11279,T. Hrin,1537418400,MILW,CH,L,30,7.808169,-7.013686,6.739222,1.304331,5.749998,76.989118,1108.109221,0.207911,45
3511,D. Terwilliger,1306774273,MASS,SL,R,27,1.224218,-3.069099,5.708333,-4.144808,3.107571,74.039018,2213.422137,0.201261,27


In [13]:
stuff.sort_values("xrv").head(10)

,Pitcher,pitcherId,Team,PitchType,PitcherThrows,Count,inducedVertBreak,horzBreak,extension,relX,relZ,releaseVelocity,spinRate,xrv,Stuff+
4048,L. Earnhardt,1160764160,WIN,FS,L,78,1.793758,-6.601116,5.293057,1.278459,6.489215,77.953334,1084.659355,-0.191950,135
10285,B. Bak,1123967488,UIC,FC,L,33,3.833749,-1.239232,4.741356,0.977097,6.242619,79.076230,2258.488391,-0.090345,146
4768,J. Vanesko,1210138624,BRY,FC,L,41,9.463535,-1.680251,6.344392,1.777137,6.043031,81.595356,2183.780640,-0.082906,144
11650,G. Helsel,1162012672,XAV,FC,L,133,7.694635,7.974274,5.365624,2.512354,6.612456,78.298054,2181.082149,-0.066783,140
12254,K. Sweum,1558099259,GONZ,FS,L,35,5.326969,-3.456027,5.469774,1.449295,6.248893,81.543525,1070.602571,-0.065139,120
6753,A. Duncan,1459122262,DBU,FS,R,103,-2.005443,11.499934,6.151737,-2.631159,4.938503,85.095317,1398.422058,-0.054293,118
4885,D. Volantis,1979617275,TEX,FC,L,191,0.282130,-1.573812,5.424475,2.184561,6.770794,86.562406,2558.181673,-0.047219,134
8288,J. Noot,1142711296,LSU,FS,R,38,-0.118617,16.766627,4.886609,-1.854534,6.168372,87.843023,1514.607407,-0.038562,116
12402,T. Beard,1100603904,FSU,FS,L,39,4.747969,-4.393101,5.459938,0.824600,6.530789,83.396143,1445.272831,-0.038306,116
2753,E. Maloney,1085415168,URI,CH,R,87,9.350467,9.442642,4.927112,-0.090336,6.935631,75.586409,1500.601278,-0.034565,162


In [14]:
stuff.to_csv("stuff.csv")

In [15]:
stuff[stuff["Count"] > 200].sort_values("Stuff+", ascending=False)

,Pitcher,pitcherId,Team,PitchType,PitcherThrows,Count,inducedVertBreak,horzBreak,extension,relX,relZ,releaseVelocity,spinRate,xrv,Stuff+
11050,M. Knight,1304692992,MOSU,CU,L,292,-12.177071,16.481885,5.526359,1.782255,6.185066,77.952928,2984.963765,0.005000,133
4823,B. Frederick,1158279680,TENN,FA,R,297,1.572949,7.436813,5.701303,-2.901594,1.640074,82.359681,2168.243243,0.038908,133
5785,M. Brassfield,1038792681,TCU,SL,L,344,-0.467998,7.379557,5.455084,1.921001,6.240316,84.950691,2751.271528,0.021693,132
2388,J. Raab,1139725824,GTWN,FA,R,207,23.336993,11.135619,6.086409,-2.208609,6.104525,92.491013,2511.967254,0.041329,132
6433,H. Dietz,1301090561,ARK,FC,L,296,0.792318,2.065519,5.576978,2.386098,6.519749,85.249631,2484.551708,-0.033575,131
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1516,B. Moulin,1271709434,PENN,SI,L,227,3.457303,-16.990851,5.452604,3.149611,5.076557,85.097481,2063.482848,0.187796,77
5528,L. Mercurius,1092978944,OKLA,CH,R,251,10.246654,13.497559,6.369582,-1.553438,5.811309,83.556424,1976.030050,0.140561,77
6424,C. Fisher,1304693248,ARK,FA,L,229,7.410462,-13.283962,5.502587,2.210741,5.854535,91.077974,2316.451911,0.172124,76
5249,J. Ruller,1306199808,QUC,SI,L,262,11.472739,-17.682614,5.621919,2.033842,5.477155,87.412786,2128.927200,0.202985,72


In [18]:
stuff[(stuff["Team"] == "URI")].sort_values("Stuff+", ascending=False).head(60)

,Pitcher,pitcherId,Team,PitchType,PitcherThrows,Count,inducedVertBreak,horzBreak,extension,relX,relZ,releaseVelocity,spinRate,xrv,Stuff+
2753,E. Maloney,1085415168,URI,CH,R,87,9.350467,9.442642,4.927112,-0.090336,6.935631,75.586409,1500.601278,-0.034565,162
3196,J. Cullen,1358089441,URI,FA,R,280,21.846538,10.306399,6.014419,-1.058248,6.122621,91.539084,2256.348520,0.067374,121
3667,C. Grotyohann,1203468032,URI,FA,R,135,21.439212,9.142326,5.930859,-1.855560,6.192487,90.682450,2188.439826,0.070641,119
2754,E. Maloney,1085415168,URI,CU,R,60,-6.452983,-1.332604,4.757064,-0.271923,6.784984,73.773580,1926.535019,0.038602,117
3211,J. Sabbath,1110853376,URI,SL,R,206,5.095956,-11.401671,5.865138,-1.798176,5.668853,78.623431,2351.838092,0.052849,114
3206,A. Jones,1101042688,URI,FA,R,105,21.236874,7.910745,5.901983,-1.152742,5.851122,89.478588,2143.883681,0.092205,110
2762,C. Johnston,1241946624,URI,CH,R,119,10.168473,14.196056,6.480022,-0.716080,6.055721,79.984805,1727.809956,0.072422,110
3209,J. Sabbath,1110853376,URI,FA,R,321,18.185708,2.550639,6.510637,-1.570349,5.790900,89.892007,2227.848003,0.096959,108
3210,J. Sabbath,1110853376,URI,CH,R,41,4.957220,10.890884,6.762260,-1.949564,5.575793,84.669759,1628.210072,0.077542,108
10543,S. Houchens,1194448803,URI,FA,L,212,18.949416,-11.027003,5.505743,1.666993,6.225490,87.991802,2091.028407,0.095930,108


In [19]:
stuff[(stuff["Count"] > 50) & (stuff["PitchType"] == "FA")].sort_values("Stuff+", ascending=False).head(10)

,Pitcher,pitcherId,Team,PitchType,PitcherThrows,Count,inducedVertBreak,horzBreak,extension,relX,relZ,releaseVelocity,spinRate,xrv,Stuff+
490,J. Music,1161348114,CAMP,FA,R,173,21.628388,13.648429,5.659460,-3.559814,5.955501,91.997280,2213.872287,0.034663,135
4823,B. Frederick,1158279680,TENN,FA,R,297,1.572949,7.436813,5.701303,-2.901594,1.640074,82.359681,2168.243243,0.038908,133
11971,J. Nottingham,1557710177,UGA,FA,R,145,21.125900,12.154075,6.015362,-1.536138,6.166429,95.914997,2560.459596,0.038500,133
2388,J. Raab,1139725824,GTWN,FA,R,207,23.336993,11.135619,6.086409,-2.208609,6.104525,92.491013,2511.967254,0.041329,132
2281,C. West,1310862081,CONN,FA,L,399,23.521863,-5.871947,6.022545,0.330315,6.209742,90.623788,2244.151307,0.041957,131
6190,M. Mueller,1260830208,MILW,FA,L,78,22.007700,-10.547277,6.080119,0.308997,6.917118,89.385038,2299.307681,0.042998,131
4345,S. Sandford,1464054215,FLA,FA,R,192,17.901868,10.574753,6.306502,-3.370593,5.863227,95.055346,2300.540448,0.046436,130
8274,S. Garcia,1602557566,LSU,FA,L,148,21.411937,-11.063595,6.142356,1.607598,6.253821,92.751165,2448.229255,0.044807,130
2653,W. Whelan,1162337024,MINN,FA,L,111,15.263418,-16.435863,5.736838,3.669298,5.293561,92.051269,2247.017519,0.047387,129
9769,C. Powell,1041792638,TXAM,FA,L,82,19.423501,-11.942774,6.072767,2.334765,5.699053,92.385634,2359.194897,0.048057,129


In [20]:
stuff[(stuff["PitcherThrows"] == "R") & (stuff["PitchType"] == "FA")].sort_values("Stuff+", ascending=False).head(10)

,Pitcher,pitcherId,Team,PitchType,PitcherThrows,Count,inducedVertBreak,horzBreak,extension,relX,relZ,releaseVelocity,spinRate,xrv,Stuff+
490,J. Music,1161348114,CAMP,FA,R,173,21.628388,13.648429,5.659460,-3.559814,5.955501,91.997280,2213.872287,0.034663,135
11971,J. Nottingham,1557710177,UGA,FA,R,145,21.125900,12.154075,6.015362,-1.536138,6.166429,95.914997,2560.459596,0.038500,133
4823,B. Frederick,1158279680,TENN,FA,R,297,1.572949,7.436813,5.701303,-2.901594,1.640074,82.359681,2168.243243,0.038908,133
2388,J. Raab,1139725824,GTWN,FA,R,207,23.336993,11.135619,6.086409,-2.208609,6.104525,92.491013,2511.967254,0.041329,132
4345,S. Sandford,1464054215,FLA,FA,R,192,17.901868,10.574753,6.306502,-3.370593,5.863227,95.055346,2300.540448,0.046436,130
10072,C. Clark,1956935142,USM,FA,R,331,21.549512,9.081232,6.479228,-1.775685,5.490115,93.422269,2266.997569,0.048969,129
4355,J. Barberi,1862463014,FLA,FA,R,212,20.894572,10.942414,5.447520,-1.818514,6.297267,97.477952,2525.121508,0.047150,129
6375,A. Roblez,1180787200,ORST,FA,R,118,24.373893,2.416431,6.217429,-0.757470,6.694467,91.422169,2342.904456,0.050728,128
1894,D. Sheerin,1213263616,LSU,FA,R,309,14.685014,16.386464,6.015708,-3.526091,5.499113,96.379109,2446.678971,0.053563,127
10565,L. ODonnell,1691990270,USF,FA,R,97,21.633855,9.905414,6.210631,-1.165482,6.300660,95.154776,2477.496614,0.052742,127


In [21]:
import pandas as pd
import numpy as np
stuff = pd.read_csv("stuff.csv")
pitches = list(stuff["PitchType"].unique())

In [22]:
rows = []
ids = list(stuff["pitcherId"].unique())

for i in ids:
    x = stuff[stuff["pitcherId"] == i]
    dx = {}
    dx["Pitcher"] = x["Pitcher"].iloc[0]
    dx["pitcherId"] = i
    dx["Team"] = x["Team"].iloc[0]
    dx["PitcherThrows"] = x["PitcherThrows"].iloc[0]
    for pt in list(stuff["PitchType"].unique()):
        if pt in list(x["PitchType"].unique()):
            dx["Stf+ " + pt] = x[x["PitchType"] == pt]["Stuff+"].iloc[0].astype("int64")
            dx[pt + "_ct"] = x[x["PitchType"] == pt]["Count"].iloc[0]
        else:
            dx["Stf+ " + pt] = np.nan
    dx["Total"] = x["Count"].sum()
    rows.append(dx)

In [23]:
table = pd.DataFrame(rows).reset_index(drop=True)
stf_cols = [col for col in table.columns if col.startswith("Stf+")]
table[stf_cols] = table[stf_cols].astype("Int64")

table["Stuff+"] = 0.0
for p in pitches:
    col = "Stf+ " + p
    ct_col = p + "_ct"
    if col in table.columns and ct_col in table.columns:
        mask = table[col].notna()
        table.loc[mask, "Stuff+"] += (
            table.loc[mask, col] * (table.loc[mask, ct_col] / table.loc[mask, "Total"])
        )

table["Stuff+"] = table["Stuff+"].round().astype(int)
table = table[["Pitcher", "pitcherId", "Team", "Total", "PitcherThrows", "Stf+ FA", "Stf+ SL", "Stf+ CU", "Stf+ FC", "Stf+ SI", "Stf+ CH", "Stf+ FS", "Stuff+"]]

In [24]:
table[table["Team"] == "URI"].sort_values("Stuff+", ascending=False)

,Pitcher,pitcherId,Team,Total,PitcherThrows,Stf+ FA,Stf+ SL,Stf+ CU,Stf+ FC,Stf+ SI,Stf+ CH,Stf+ FS,Stuff+
880,E. Maloney,1085415168,URI,357,R,101,<NA>,117,<NA>,<NA>,162,<NA>,119
1187,C. Grotyohann,1203468032,URI,200,R,119,<NA>,<NA>,101,<NA>,<NA>,<NA>,113
1034,J. Cullen,1358089441,URI,597,R,121,107,106,101,<NA>,96,<NA>,111
1039,J. Sabbath,1110853376,URI,630,R,108,114,97,<NA>,<NA>,108,<NA>,109
4398,W. Creed,1299086745,URI,75,L,107,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,107
881,C. Maloney,1336247552,URI,156,R,107,105,<NA>,<NA>,<NA>,<NA>,<NA>,106
1038,A. Jones,1101042688,URI,174,R,110,96,99,<NA>,<NA>,<NA>,<NA>,105
1037,M. Santos,1487988943,URI,326,R,103,104,<NA>,<NA>,<NA>,<NA>,<NA>,103
3529,S. Houchens,1194448803,URI,466,L,108,101,<NA>,<NA>,<NA>,82,<NA>,103
1036,B. Perry,1342168540,URI,38,R,102,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,102


In [26]:
table[table["Total"] >= 500].sort_values("Stuff+", ascending=False).head(10)

,Pitcher,pitcherId,Team,Total,PitcherThrows,Stf+ FA,Stf+ SL,Stf+ CU,Stf+ FC,Stf+ SI,Stf+ CH,Stf+ FS,Stuff+
2119,H. Dietz,1301090561,ARK,998,L,125,122,133,131,<NA>,<NA>,<NA>,127
3508,C. Cavanaugh,1196796416,DAV,641,L,127,<NA>,130,<NA>,<NA>,119,<NA>,126
759,R. Marohn,1197348608,NCST,638,L,125,<NA>,125,<NA>,<NA>,125,<NA>,125
3492,W. Sanford,1518897568,ORE,991,R,125,100,116,<NA>,<NA>,<NA>,<NA>,121
1414,L. Peterson,1142994176,FLA,1042,R,126,119,105,<NA>,<NA>,113,<NA>,120
2524,C. Airington,1041288587,PEAY,613,R,123,97,<NA>,<NA>,<NA>,<NA>,105,119
1014,M. Stammel,1489749081,UVA,791,L,122,107,<NA>,121,<NA>,116,<NA>,119
2603,E. Nachtsheim,1256101932,MCNS,837,R,126,101,103,92,<NA>,97,<NA>,119
1947,J. Harper,1700152819,CSF,561,L,127,101,<NA>,<NA>,<NA>,99,<NA>,119
4137,K. Sweum,1558099259,GONZ,918,L,122,118,<NA>,114,<NA>,95,120,119


In [27]:
table[table["Total"] >= 100].sort_values("Stuff+", ascending=False).head(10)

,Pitcher,pitcherId,Team,Total,PitcherThrows,Stf+ FA,Stf+ SL,Stf+ CU,Stf+ FC,Stf+ SI,Stf+ CH,Stf+ FS,Stuff+
4025,J. Nottingham,1557710177,UGA,145,R,133,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,133
841,W. Whelan,1162337024,MINN,111,L,129,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,129
2119,H. Dietz,1301090561,ARK,998,L,125,122,133,131,<NA>,<NA>,<NA>,127
150,J. Music,1161348114,CAMP,255,R,135,125,<NA>,<NA>,<NA>,96,<NA>,127
3508,C. Cavanaugh,1196796416,DAV,641,L,127,<NA>,130,<NA>,<NA>,119,<NA>,126
2336,S. Fitzpatrick,1082403072,ASU,310,L,128,124,<NA>,<NA>,<NA>,<NA>,<NA>,126
762,J. Raab,1139725824,GTWN,268,R,132,103,<NA>,<NA>,<NA>,<NA>,104,125
759,R. Marohn,1197348608,NCST,638,L,125,<NA>,125,<NA>,<NA>,125,<NA>,125
1583,B. Frederick,1158279680,TENN,398,R,133,98,<NA>,<NA>,<NA>,<NA>,<NA>,124
4088,N. Helman,1756665469,KENN,376,R,123,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,123


In [28]:
table[table["Total"] >= 100].sort_values("Stf+ FS", ascending=False).head(10)

,Pitcher,pitcherId,Team,Total,PitcherThrows,Stf+ FA,Stf+ SL,Stf+ CU,Stf+ FC,Stf+ SI,Stf+ CH,Stf+ FS,Stuff+
1316,L. Earnhardt,1160764160,WIN,705,L,108,96,109,<NA>,<NA>,114,135,109
4137,K. Sweum,1558099259,GONZ,918,L,122,118,<NA>,114,<NA>,95,120,119
2228,A. Duncan,1459122262,DBU,284,R,<NA>,<NA>,<NA>,<NA>,111,<NA>,118,114
2718,J. Noot,1142711296,LSU,117,R,93,<NA>,<NA>,<NA>,95,<NA>,116,101
4193,T. Beard,1100603904,FSU,834,L,111,76,122,<NA>,<NA>,102,116,105
1410,A. King,1929502555,FLA,950,R,111,102,<NA>,<NA>,<NA>,91,114,107
1547,P. Bauer,1791552299,JMU,658,R,100,95,99,<NA>,<NA>,<NA>,114,99
3585,N. Mertens,1643596490,SEMO,582,R,97,<NA>,<NA>,<NA>,<NA>,109,113,101
4242,J. Price,1342170246,EKY,947,R,84,97,79,100,98,103,112,93
4056,J. Jones,1322928128,CARK,288,R,103,<NA>,90,<NA>,<NA>,104,112,103


In [29]:
table[table["Team"] == "MRMK"].sort_values("Stuff+", ascending=False)

,Pitcher,pitcherId,Team,Total,PitcherThrows,Stf+ FA,Stf+ SL,Stf+ CU,Stf+ FC,Stf+ SI,Stf+ CH,Stf+ FS,Stuff+
590,Z. Broderick,1327271424,MRMK,128,L,104,<NA>,<NA>,<NA>,<NA>,95,<NA>,102
589,S. Franco,1769668118,MRMK,75,L,95,<NA>,105,<NA>,<NA>,<NA>,<NA>,99
562,C. Kiesiner,1402009816,MRMK,117,R,98,97,<NA>,<NA>,<NA>,<NA>,<NA>,98
560,C. Barker,1304701440,MRMK,189,R,95,90,<NA>,107,<NA>,<NA>,<NA>,95
568,O. Barkal,1384801355,MRMK,183,R,84,97,<NA>,<NA>,<NA>,<NA>,<NA>,95
564,N. Hunkele,1342167952,MRMK,223,R,95,83,<NA>,<NA>,<NA>,102,<NA>,94
561,B. Loper,1307046400,MRMK,90,R,87,83,<NA>,<NA>,<NA>,110,<NA>,92
565,J. Nichols,1907374041,MRMK,153,R,87,98,<NA>,96,<NA>,<NA>,<NA>,92
747,J. Borsari,1213486080,MRMK,71,R,90,93,<NA>,<NA>,<NA>,<NA>,<NA>,92
748,O. Zadrozny,1260854784,MRMK,55,R,92,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,92


In [30]:
stuff[stuff["pitcherId"] == 1304701440]

,Unnamed: 0,Pitcher,pitcherId,Team,PitchType,PitcherThrows,Count,inducedVertBreak,horzBreak,extension,relX,relZ,releaseVelocity,spinRate,xrv,Stuff+
1771,1771,C. Barker,1304701440,MRMK,FA,R,87,17.802189,11.652672,6.193289,-1.426901,6.342836,88.475359,2141.249840,0.126433,95
1772,1772,C. Barker,1304701440,MRMK,SL,R,70,-0.921824,-5.590443,5.940046,-1.555277,6.218094,77.022350,2132.531281,0.094890,90
1773,1773,C. Barker,1304701440,MRMK,FC,R,32,8.375591,0.372366,6.197989,-1.556288,6.185961,82.083254,1992.432315,0.053412,107


In [31]:
table.to_csv("stuff_table.csv")

In [32]:
table[table["Team"] == "ECU"].sort_values("Stuff+", ascending=False)

,Pitcher,pitcherId,Team,Total,PitcherThrows,Stf+ FA,Stf+ SL,Stf+ CU,Stf+ FC,Stf+ SI,Stf+ CH,Stf+ FS,Stuff+
3574,C. Weber,1369039521,ECU,215,R,120,119,<NA>,<NA>,<NA>,<NA>,<NA>,120
1209,G. Van Kempen,1099204864,ECU,428,R,113,117,<NA>,109,<NA>,101,<NA>,113
1086,L. Payne,1093095680,ECU,305,L,113,<NA>,<NA>,<NA>,<NA>,111,<NA>,112
278,C. Hoagland,1360113278,ECU,223,L,100,103,<NA>,<NA>,119,<NA>,<NA>,111
354,G. Marley,1225307968,ECU,454,R,112,95,<NA>,<NA>,<NA>,<NA>,<NA>,111
4606,T. Paxton,1684934913,ECU,31,R,110,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,110
353,R. Towers,1196825344,ECU,502,L,112,<NA>,98,<NA>,<NA>,104,<NA>,108
1017,E. Norby,1335271936,ECU,1017,L,114,104,<NA>,90,105,111,<NA>,107
722,A. Bouche,1310415616,ECU,140,R,101,111,110,<NA>,<NA>,<NA>,<NA>,106
131,S. Jenkins,1058441697,ECU,505,R,98,109,<NA>,107,101,<NA>,<NA>,105
